# TC4 Amazon: modelagem para o Power BI

**Arquivo novo. Não substitui nem altera `tc_4_amazon.ipynb` nem `tc_4_amazon novo.ipynb`.**

Os dois notebooks do grupo entregam a limpeza da base e o NLP sobre `about_product`.
Este pega de onde eles param e resolve o que faltava para o dashboard.

Rodado contra a base real (1.465 linhas, 16 colunas) em 29/10/2026. Os números deste
cabeçalho saíram dessa execução.

| # | O quê | Evidência na base real |
|---|---|---|
| 1 | `category` vira **coluna por nível**, não linha | o `explode` leva 1.351 produtos a 5.786 linhas e infla `SUM(rating_count)` de 23,8 mi para 98,9 mi, **4,2 vezes** |
| 2 | os IDs repetidos ganham **regra de desempate** | 92 IDs em disputa: 47 divergem só em link ou categoria, **45 divergem em dado**, a maioria em `rating_count` |
| 3 | `review_title` e `review_content` passam a ser **tratados** | **10.598 avaliações** em 1.351 produtos, média de 7,8 cada. Nenhum notebook tocava nessas colunas |
| 4 | **`dMoeda`** desconectada | a base só tem rupia, e a conversão precisa de premissa declarada |

---

## NOTA 1: por que a categoria não pode virar linha

A coluna `category` da base é assim:

```
Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables
```

É **uma categoria só, escrita como caminho hierárquico**, não cinco categorias. O `|`
separa nível, não separa item. A base tem até **7 níveis**, mas eles afunilam rápido:

| Nível | Caminhos preenchidos |
|---|---|
| 1 e 2 | 211 |
| 3 | 207 |
| 4 | 174 |
| 5 | 68 |
| 6 | 19 |
| 7 | 7 |

Na prática o drill-down útil vai até o nível 3 ou 4. Do 5 em diante a amostra não sustenta
leitura.

Quando o notebook do grupo roda `str.split('|')` seguido de `explode`, cada produto vira
uma linha por nível. Num modelo de Power BI em cima disso, `SUM(rating_count)` devolve 4,2
vezes o valor real, `COUNTROWS` conta errado e `AVERAGE(rating)` fica ponderado pela
profundidade da categoria em vez de pelo produto.

## NOTA 2: por que o corte de percepção é percentil, não número fixo

O `rating` da base vai de **2,0 a 5,0**, com mediana **4,10**:

| | p5 | p10 | p25 | p50 | p75 | p90 |
|---|---|---|---|---|---|---|
| rating | 3,60 | 3,70 | 3,90 | 4,10 | 4,30 | 4,40 |

Um corte fixo em `rating < 3,0` marca **6 produtos de 1.351** como negativos. Com seis
produtos de um lado e mil do outro não existe comparação de padrão em avaliação positiva
contra negativa, que é exatamente o que o enunciado pede.

O corte por percentil compara cada produto com o resto da própria base. Com p25 e p75:
**226 abaixo, 706 na média, 419 acima**. Aí dá para comparar.

Isso também é um achado por si só, e cabe no vídeo: **a base não tem produto mal avaliado
em termos absolutos.** Tudo que parece ruim só é ruim em relação aos vizinhos.

## NOTA 3: `review_id` não é chave

São 10.598 linhas de avaliação para **9.265 `review_id` distintos**. Os 759 ids repetidos
aparecem sempre em **produtos diferentes**, nunca duas vezes no mesmo produto: é a mesma
avaliação ligada a variações do mesmo item.

Usar `review_id` como chave quebraria o relacionamento no Power BI. A tabela tem chave
substituta `sk_avaliacao`, e `review_id` fica como atributo.

## 1. Carregar e limpar

Mesma lógica de tipagem do notebook do grupo, reorganizada em função.

In [ ]:
import re
import pandas as pd

CAMINHO = '/content/amazon.csv'   # trocar pelo caminho local se não for Colab

bruto = pd.read_csv(CAMINHO, sep=',')
print(f'base original: {len(bruto)} linhas, {bruto.shape[1]} colunas')
bruto.head(3)

In [ ]:
COLS_PRECO = ['discounted_price_rupia', 'actual_price_rupia']
COLS_LINK = ['img_link', 'product_link']


def limpar(df: pd.DataFrame) -> pd.DataFrame:
    """Tipagem e normalizacao. Mesma logica do notebook do grupo."""
    df = df.copy()
    df.columns = df.columns.str.strip()
    df = df.replace(r"^\s*$", pd.NA, regex=True)
    df = df.rename(columns={'discounted_price': 'discounted_price_rupia',
                            'actual_price': 'actual_price_rupia'})
    for c in COLS_PRECO:
        df[c] = (df[c].astype('string')
                 .str.replace('₹', '', regex=False)
                 .str.replace(',', '', regex=False).str.strip())
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df['discount_percentage'] = pd.to_numeric(
        df['discount_percentage'].astype('string').str.replace('%', '', regex=False),
        errors='coerce') / 100
    df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
    df['rating_count'] = pd.to_numeric(
        df['rating_count'].astype('string').str.replace(',', '', regex=False),
        errors='coerce')
    for c in df.select_dtypes(include=['object', 'string']).columns:
        df[c] = df[c].astype('string').str.strip()
    return df

In [ ]:
limpo = limpar(bruto)
print(f'produtos únicos na base limpa: {limpo["product_id"].nunique()}')   # esperado: 1351
limpo[COLS_PRECO + ['discount_percentage', 'rating', 'rating_count']].describe()

## 2. Um product_id, uma linha

O notebook do grupo consolida os IDs que divergem só em link e categoria e deixa os outros
duplicados. Aqui todos ganham desempate, e o descartado vai para relatório, não some calado.

In [ ]:
def resolver_duplicados(df: pd.DataFrame):
    """
    Um product_id tem que virar uma linha so. Regra, nessa ordem:
      1. menos campos nulos vence
      2. empate: maior rating_count vence
      3. empate: primeira ocorrencia
    Devolve (df_unico, relatorio) com tudo o que foi descartado.
    """
    df = df.copy()
    chave = 'product_id'
    df['_nulos'] = df.isna().sum(axis=1)
    df['_ordem'] = range(len(df))
    df = df.sort_values([chave, '_nulos', 'rating_count', '_ordem'],
                        ascending=[True, True, False, True])

    repetidos = df[df.duplicated(subset=chave, keep=False)]
    comparaveis = [c for c in df.columns
                   if c not in [chave, '_nulos', '_ordem'] + COLS_LINK + ['category']]
    relatorio = []
    for pid, g in repetidos.groupby(chave):
        divergentes = [c for c in comparaveis if g[c].nunique(dropna=False) > 1]
        relatorio.append({
            'product_id': pid,
            'registros': len(g),
            'colunas_divergentes': ', '.join(divergentes) if divergentes
                                   else '(so links e/ou categoria)',
            'qtd_colunas_divergentes': len(divergentes),
            'mantido_rating_count': g.iloc[0]['rating_count'],
            'mantido_nulos': int(g.iloc[0]['_nulos']),
        })

    unico = df.drop_duplicates(subset=chave, keep='first').drop(columns=['_nulos', '_ordem'])
    return unico.reset_index(drop=True), pd.DataFrame(relatorio)

In [ ]:
unico, relatorio_duplicados = resolver_duplicados(limpo)
print(f'{len(limpo)} linhas -> {len(unico)} produtos')
print(f'IDs que precisaram de desempate: {len(relatorio_duplicados)}')   # esperado: 92
so_link = (relatorio_duplicados.colunas_divergentes == '(so links e/ou categoria)').sum()
print(f'   divergiam só em link ou categoria: {so_link}')
print(f'   divergiam em coluna de dado:       {len(relatorio_duplicados) - so_link}')
display(relatorio_duplicados.sort_values('qtd_colunas_divergentes', ascending=False).head(10))

## 3. Categoria: hierarquia em colunas

Ver a NOTA 1 no topo.

In [ ]:
def dividir_categoria(df: pd.DataFrame, coluna='category'):
    """
    `category` e UMA hierarquia separada por `|`, nao uma lista de categorias.
    Entao vira coluna por nivel, nao linha por nivel. Devolve (dCategoria, de_para).
    """
    caminhos = (df[coluna].astype('string').fillna('(sem categoria)')
                .str.split('|').apply(lambda ns: [n.strip() for n in ns if str(n).strip()]))
    profundidade = int(caminhos.apply(len).max())

    dim = pd.DataFrame({'caminho_categoria': df[coluna].astype('string')
                        .fillna('(sem categoria)')})
    for i in range(profundidade):
        dim[f'nivel_{i+1}'] = caminhos.apply(lambda ns, i=i: ns[i] if len(ns) > i else pd.NA)
    dim['categoria_folha'] = caminhos.apply(lambda ns: ns[-1] if ns else pd.NA)
    dim['profundidade'] = caminhos.apply(len)

    dim = dim.drop_duplicates(subset='caminho_categoria').reset_index(drop=True)
    dim.insert(0, 'sk_categoria', range(1, len(dim) + 1))

    de_para = (df[['product_id', coluna]].assign(
        caminho_categoria=df[coluna].astype('string').fillna('(sem categoria)'))
        .merge(dim[['sk_categoria', 'caminho_categoria']], on='caminho_categoria')
        [['product_id', 'sk_categoria']])
    return dim, de_para

In [ ]:
dCategoria, de_para_categoria = dividir_categoria(unico)
niveis = [x for x in dCategoria.columns if x.startswith('nivel_')]
print(f'caminhos distintos: {len(dCategoria)} | profundidade máxima: {len(niveis)} níveis')
print(f'de_para: {len(de_para_categoria)} linhas (tem que bater com os {len(unico)} produtos)')
print()
print('preenchimento por nível:')
for n in niveis:
    print(f'   {n}: {dCategoria[n].notna().sum()}')
print()
print('produtos por categoria de nível 1:')
print(dCategoria.merge(de_para_categoria, on='sk_categoria').nivel_1.value_counts().to_string())

## 4. Avaliações: o que ninguém tinha pego

`review_id` e `user_id` são listas separadas por vírgula e sem vírgula no conteúdo, então
dá para separar com segurança. Já `review_title` e `review_content` também vêm colados por
vírgula, **mas o texto tem vírgula dentro**, o que torna o split não confiável.

Decisão declarada: a contagem sai do `review_id`, e o texto fica agregado por produto para
a extração de termos. Ver também a NOTA 3 sobre a chave.

In [ ]:
def separar_avaliacoes(df: pd.DataFrame):
    """
    `review_id` e `user_id` sao listas separadas por virgula e limpas.
    `review_title` e `review_content` tambem sao coladas por virgula, mas o texto
    tem virgula dentro, entao dividir por virgula corrompe o conteudo.
    Decisao: contar avaliacoes pelo review_id, e manter o texto agregado por produto.
    """
    def pedacos(v):
        if pd.isna(v):
            return []
        return [p.strip() for p in str(v).split(',') if p.strip()]

    linhas = []
    for _, r in df.iterrows():
        ids = pedacos(r.get('review_id'))
        usuarios = pedacos(r.get('user_id'))
        for i, rid in enumerate(ids):
            linhas.append({
                'review_id': rid,
                'product_id': r['product_id'],
                'user_id': usuarios[i] if i < len(usuarios) else pd.NA,
                'posicao': i + 1,
            })
    fato = pd.DataFrame(linhas, columns=['review_id', 'product_id', 'user_id', 'posicao'])
    # `review_id` NAO e chave: o mesmo id aparece em produtos diferentes (variacoes do
    # mesmo item). Entao a chave da tabela e substituta, e review_id vira atributo.
    fato.insert(0, 'sk_avaliacao', range(1, len(fato) + 1))

    texto = df[['product_id', 'review_title', 'review_content', 'rating', 'rating_count']].copy()
    texto['qtd_avaliacoes_no_campo'] = texto['product_id'].map(
        fato.groupby('product_id').size()).fillna(0).astype(int)
    return fato, texto


STOPWORDS_EN = {
 'the','and','for','that','this','with','you','its','are','was','have','has','but','not',
 'all','can','from','very','good','just','one','out','use','used','using','will','they',
 'your','our','their','there','when','what','who','how','than','then','them','these','those',
 'been','being','does','did','doing','into','over','under','more','most','some','such','only',
 'own','same','too','also','get','got','had','her','his','him','she','him','its','it','is','a',
 'an','of','to','in','on','at','as','by','or','if','so','we','be','do','no','my','me','up','i',
 # genericos do dominio: aparecem em tudo e nao separam produto bom de ruim.
 # Lista montada olhando os 30 termos mais frequentes da base real.
 'product','products','quality','price','buy','bought','item','amazon','nice','good','best',
 'like','really','well','after','which','value','money','time','overall','other','better',
 'even','any','work','works','working','use','useful','bit','one','two','day','days','month',
 'months','year','years','much','many','lot','little','still','make','made','need','needs',
 'want','going','come','comes','take','takes','give','gives','see','say','says','know','think'
}

In [ ]:
fAvaliacoes, texto_avaliacoes = separar_avaliacoes(unico)
print(f'avaliações: {len(fAvaliacoes):,} linhas')              # esperado: 10.598
print(f'review_id distintos: {fAvaliacoes.review_id.nunique():,}')   # esperado: 9.265
print(f'média por produto: {len(fAvaliacoes) / len(unico):.1f}')
display(fAvaliacoes.head())

### 4.1 Termos por percepção

Ver a NOTA 2 no topo: o corte é percentil da própria base, não número fixo.

In [ ]:
def limpar_texto(t) -> str:
    if pd.isna(t):
        return ''
    t = re.sub(r'<[^>]+>', ' ', str(t))
    t = re.sub(r'http\S+', ' ', t)
    t = t.lower()
    t = re.sub(r'[^a-z\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

In [ ]:
STOPWORDS_EN = {
 'the','and','for','that','this','with','you','its','are','was','have','has','but','not',
 'all','can','from','very','good','just','one','out','use','used','using','will','they',
 'your','our','their','there','when','what','who','how','than','then','them','these','those',
 'been','being','does','did','doing','into','over','under','more','most','some','such','only',
 'own','same','too','also','get','got','had','her','his','him','she','him','its','it','is','a',
 'an','of','to','in','on','at','as','by','or','if','so','we','be','do','no','my','me','up','i',
 # genericos do dominio: aparecem em tudo e nao separam produto bom de ruim.
 # Lista montada olhando os 30 termos mais frequentes da base real.
 'product','products','quality','price','buy','bought','item','amazon','nice','good','best',
 'like','really','well','after','which','value','money','time','overall','other','better',
 'even','any','work','works','working','use','useful','bit','one','two','day','days','month',
 'months','year','years','much','many','lot','little','still','make','made','need','needs',
 'want','going','come','comes','take','takes','give','gives','see','say','says','know','think'
}

In [ ]:
def faixas_de_percepcao(texto: pd.DataFrame, q_baixo=0.25, q_alto=0.75):
    """
    Devolve os dois cortes de nota que separam percepcao, em PERCENTIL da propria base.

    Por que nao corte fixo: na base da Amazon o rating vai de 2,0 a 5,0, com mediana
    4,10 e p5 em 3,60. Um corte fixo em 3,0 marca 6 produtos de 1.351 como negativos,
    o que inviabiliza qualquer comparacao. Percentil compara cada produto com o resto
    da propria base, que e a pergunta que o negocio faz.
    """
    notas = texto['rating'].dropna()
    return float(notas.quantile(q_baixo)), float(notas.quantile(q_alto))

In [ ]:
def termos_das_avaliacoes(texto: pd.DataFrame, q_baixo=0.25, q_alto=0.75,
                          minimo_letras=3) -> pd.DataFrame:
    """
    Frequencia de termo por produto e por percepcao.
    A percepcao vem do `rating`, porque a base nao traz rotulo de sentimento.
    PREMISSA DECLARADA, e ela precisa aparecer no dashboard e no video.
    """
    corte_baixo, corte_alto = faixas_de_percepcao(texto, q_baixo, q_alto)

    def faixa(r):
        if pd.isna(r):
            return 'sem nota'
        if r >= corte_alto:
            return 'acima da base'
        if r <= corte_baixo:
            return 'abaixo da base'
        return 'na media'

    linhas = []
    for _, r in texto.iterrows():
        def ou_vazio(v):
            return '' if pd.isna(v) else str(v)
        bruto = ou_vazio(r.get('review_title')) + ' ' + ou_vazio(r.get('review_content'))
        percepcao = faixa(r['rating'])
        vistos = {}
        for termo in limpar_texto(bruto).split():
            if len(termo) < minimo_letras or termo in STOPWORDS_EN:
                continue
            vistos[termo] = vistos.get(termo, 0) + 1
        for termo, n in vistos.items():
            linhas.append({'product_id': r['product_id'], 'termo': termo,
                           'ocorrencias': n, 'percepcao': percepcao})
    return pd.DataFrame(linhas, columns=['product_id', 'termo', 'ocorrencias', 'percepcao'])

In [ ]:
corte_baixo, corte_alto = faixas_de_percepcao(texto_avaliacoes)
print(f'PREMISSA: abaixo da base <= {corte_baixo:.2f} | acima da base >= {corte_alto:.2f}')
print('(percentis 25 e 75 do rating desta base. Declarar isso no dashboard e no vídeo.)')
print()

dTermosAvaliacao = termos_das_avaliacoes(texto_avaliacoes)
print(f'pares termo/produto: {len(dTermosAvaliacao):,}')
print()
print('produtos por faixa:')
print(dTermosAvaliacao.drop_duplicates('product_id').percepcao.value_counts().to_string())
print()
for faixa in ['acima da base', 'abaixo da base']:
    top = (dTermosAvaliacao[dTermosAvaliacao.percepcao == faixa]
           .groupby('termo')['ocorrencias'].sum().sort_values(ascending=False).head(15))
    print(f'--- 15 termos mais frequentes em "{faixa}" ---')
    print(top.to_string())
    print()

## 4.2 Termos comparados entre as faixas

O corte por termo **exclusivo** de uma faixa não serve: o que aparece só de um lado é quase
sempre **nome de marca** (`boult`, `iqoo`, `sandisk`), não opinião.

O que separa de verdade é a **frequência relativa** de termos presentes nos dois lados. Um
termo que aparece em 24% dos produtos mal avaliados e em 11% dos bem avaliados carrega
sinal; um que aparece igual nos dois descreve o produto, não a satisfação.

In [ ]:
def comparar_termos(termos: pd.DataFrame, minimo_produtos=30) -> pd.DataFrame:
    """Frequencia relativa do termo em cada faixa, so para termos presentes nas duas."""
    por_faixa = termos.drop_duplicates('product_id').percepcao.value_counts()
    tot = termos.groupby(['termo', 'percepcao']).product_id.nunique().unstack(fill_value=0)
    tot = tot[(tot['acima da base'] > 0) & (tot['abaixo da base'] > 0)]   # tira nome de marca
    r = pd.DataFrame({
        'termo': tot.index,
        'pct_abaixo': (tot['abaixo da base'] / por_faixa['abaixo da base'] * 100).round(2).values,
        'pct_acima':  (tot['acima da base']  / por_faixa['acima da base']  * 100).round(2).values,
        'produtos':   tot.sum(axis=1).values})
    r = r[r.produtos >= minimo_produtos].copy()
    r['diferenca_pp'] = (r.pct_abaixo - r.pct_acima).round(2)
    r['lado'] = r.diferenca_pp.apply(lambda x: 'reclamação' if x > 0 else 'elogio')
    r['peso'] = r.diferenca_pp.abs().round(2)
    r = r.sort_values('diferenca_pp', ascending=False).reset_index(drop=True)
    r.insert(0, 'sk_termo', range(1, len(r) + 1))
    return r

def marcar_destaques(comp: pd.DataFrame, n_comparativo=8, n_lateral=15) -> pd.DataFrame:
    """Recorte de apresentacao, nao de analise. Cada grafico do painel promete um numero
    de termos no subtitulo; estas duas colunas sao o que faz o visual cumprir a promessa."""
    c = comp.copy()
    c['destaque'] = 'nao'
    c['destaque_lado'] = 'nao'
    for lado in c.lado.unique():
        do_lado = c[c.lado == lado]
        c.loc[do_lado.nlargest(n_comparativo, 'peso').index, 'destaque'] = 'sim'
        c.loc[do_lado.nlargest(n_lateral, 'peso').index, 'destaque_lado'] = 'sim'
    return c


In [ ]:
dTermoComparado = comparar_termos(dTermosAvaliacao)
print(f'termos com sinal: {len(dTermoComparado)}')
print()
print('--- mais frequente em quem está ABAIXO da base ---')
print(dTermoComparado.head(10)[['termo', 'pct_abaixo', 'pct_acima', 'diferenca_pp']].to_string(index=False))
print()
print('--- mais frequente em quem está ACIMA da base ---')
print(dTermoComparado.tail(10)[['termo', 'pct_abaixo', 'pct_acima', 'diferenca_pp']].to_string(index=False))
dTermoComparado = marcar_destaques(dTermoComparado)
print()
print(f"destaque (comparativo): {(dTermoComparado.destaque == 'sim').sum()} termos")
print(f"destaque_lado (laterais): {(dTermoComparado.destaque_lado == 'sim').sum()} termos")


## 5. dMoeda: tabela desconectada

A base só traz rupia indiana. Converter exige taxa de câmbio, que **não está na base**.

Esta tabela não se relaciona com nenhuma outra: existe para alimentar a segmentação e as
medidas via `SELECTEDVALUE`. É o mesmo padrão de tabela desconectada que a Gold do TC3 usou
na `dim_benchmark`.

**Trocar as taxas pela cotação do dia da entrega, e manter a data ao lado.**

In [ ]:
def tabela_moeda(taxas: dict, data_referencia: str) -> pd.DataFrame:
    """
    Tabela DESCONECTADA. Nao se liga a fProdutos: serve so para alimentar a
    segmentacao e as medidas com SELECTEDVALUE. A base nao traz cambio, entao
    cada taxa aqui e premissa declarada, com data.
    """
    return pd.DataFrame([
        {'sk_moeda': i + 1, 'moeda': m, 'simbolo': s, 'taxa_por_rupia': t,
         'data_referencia': data_referencia}
        for i, (m, (s, t)) in enumerate(taxas.items())
    ])

In [ ]:
TAXAS = {
    'Rupia indiana': ('₹',   1.0),
    'Real':          ('R$',  0.064),   # PREMISSA: trocar pela cotação do dia
    'Dólar':         ('US$', 0.012),   # PREMISSA: trocar pela cotação do dia
}
DATA_COTACAO = '2026-10-29'

dMoeda = tabela_moeda(TAXAS, DATA_COTACAO)
dMoeda

## 6. Tabela fato: uma linha por produto

In [ ]:
def montar_fato(df: pd.DataFrame, de_para_categoria: pd.DataFrame,
                corte_baixo: float, corte_alto: float) -> pd.DataFrame:
    """Uma linha por produto. Nada aqui pode duplicar.

    A percepcao entra aqui, no fato, com os mesmos cortes que a dTermosAvaliacao usa.
    Se ficasse so na tabela de termos, a segmentacao do painel bateria numa tabela
    sem relacionamento e nao filtraria nem os produtos nem as avaliacoes."""
    f = df[['product_id', 'product_name',
            'discounted_price_rupia', 'actual_price_rupia', 'discount_percentage',
            'rating', 'rating_count'] + COLS_LINK].copy()
    f = f.merge(de_para_categoria, on='product_id', how='left')
    f['desconto_absoluto_rupia'] = f['actual_price_rupia'] - f['discounted_price_rupia']
    f['receita_potencial_rupia'] = f['discounted_price_rupia'] * f['rating_count']
    f['percepcao'] = f['rating'].apply(
        lambda r: 'sem nota' if pd.isna(r)
        else 'acima da base' if r >= corte_alto
        else 'abaixo da base' if r <= corte_baixo
        else 'na media')
    return f

In [ ]:
fProdutos = montar_fato(unico, de_para_categoria, corte_baixo, corte_alto)
print('percepção no fato:', fProdutos.percepcao.value_counts().to_dict())
print(f'fProdutos: {len(fProdutos)} linhas')
display(fProdutos.head(3))

## 7. Validação antes de exportar

Se algum assert falhar, **não exporta**. Regra que ficou do TC3.

In [ ]:
def validar(fato, dim_cat, fato_aval, termos, bruto_limpo):
    """Asserts que tem que passar antes de exportar."""
    erros = []
    if fato['product_id'].duplicated().any():
        erros.append('fProdutos tem product_id repetido')
    if len(fato) != bruto_limpo['product_id'].nunique():
        erros.append(f"fProdutos com {len(fato)} linhas contra "
                     f"{bruto_limpo['product_id'].nunique()} ids unicos na base limpa")
    if fato['sk_categoria'].isna().any():
        erros.append('produto sem sk_categoria')
    if dim_cat['sk_categoria'].duplicated().any():
        erros.append('dCategoria com sk repetido')
    if fato_aval['sk_avaliacao'].duplicated().any():
        erros.append('fAvaliacoes com sk_avaliacao repetido')
    orfaos = set(fato_aval['product_id']) - set(fato['product_id'])
    if orfaos:
        erros.append(f'{len(orfaos)} avaliacoes sem produto correspondente')
    orfaos_t = set(termos['product_id']) - set(fato['product_id'])
    if orfaos_t:
        erros.append(f'{len(orfaos_t)} termos sem produto correspondente')
    return erros

In [ ]:
erros = validar(fProdutos, dCategoria, fAvaliacoes, dTermosAvaliacao, limpo)
if erros:
    for e in erros:
        print('ERRO:', e)
    raise SystemExit('validação falhou, não exportar')
print('validação: 0 erros')

explodido = unico.assign(category=unico.category.str.split('|')).explode('category')
print()
print(f'produtos reais:            {len(fProdutos):>12,}')
print(f'linhas se explodisse:      {len(explodido):>12,}')
print(f'SUM(rating_count) correto: {fProdutos.rating_count.sum():>12,.0f}')
print(f'SUM(rating_count) inflado: {explodido.rating_count.sum():>12,.0f}'
      f'  ({explodido.rating_count.sum() / fProdutos.rating_count.sum():.1f}x)')

## 7.5 Texto autoral: recomendações, fontes e premissas

As tabelas abaixo **não saem da base**. São a leitura que fizemos dos números calculados
acima, escritas à mão. Ficam versionadas aqui junto do resto para que qualquer pessoa
consiga ver de onde veio cada frase e refazer o painel inteiro a partir deste notebook.

No painel elas viram texto desenhado no fundo, não tabela: o número de cada recomendação
vem de uma medida do modelo e acompanha o filtro, o texto ao redor é fixo.

In [ ]:
# ATENCAO: daqui para baixo o conteudo e AUTORAL, nao extraido da base.
# Cada linha e leitura nossa sobre os numeros que as celulas acima calcularam.
# Esta na mesma pasta de saida porque o painel precisa dela, mas e texto, nao medida:
# por isso a pagina de recomendacoes mostra o texto no fundo e so o indicador vem do modelo.

dRecomendacao = pd.DataFrame([
    dict(ordem=1,
         decisao='Concentrar o sortimento em vez de diversificar',
         por_que='Três categorias concentram quase todo o catálogo. Abrir frentes novas dilui atenção onde já existe escala.',
         indicador='produtos nas 3 maiores categorias de nível 1',
         valor_indicador='1.313 de 1.351, ou 97,2%',
         onde_ver='Visão executiva'),
    dict(ordem=2,
         decisao='Parar de usar desconto para comprar avaliação',
         por_que='Desconto maior não traz mais avaliação e vem com nota levemente pior. O efeito é pequeno, mas é na direção contrária da intuição.',
         indicador='queda da nota ponderada da menor para a maior faixa de desconto',
         valor_indicador='queda de 0,14 ponto: 4,16 na faixa até 20% contra 4,01 na de 80% ou mais. Correlação desconto x nota -0,16, desconto x avaliações +0,00',
         onde_ver='Preços e descontos'),
    dict(ordem=3,
         decisao='Investigar os produtos muito avaliados com nota baixa',
         por_que='São produtos que já têm tráfego e estão decepcionando. Cada um deles converte visita em avaliação ruim.',
         indicador='produtos com nº de avaliações acima da mediana e nota no quartil inferior',
         valor_indicador='112 produtos, rating_count >= 4.740 e nota <= 3,9',
         onde_ver='Visão executiva'),
    dict(ordem=4,
         decisao='Dar visibilidade ao que já é bem avaliado e ninguém vê',
         por_que='Produto com nota alta e pouca avaliação é estoque de reputação parado. Custa menos promover do que consertar.',
         indicador='produtos com nota no quartil superior e menos de mil avaliações',
         valor_indicador='76 produtos, nota >= 4,3 e rating_count < 1.000',
         onde_ver='Visão executiva'),
    dict(ordem=5,
         decisao='Atacar ruído, bateria e material antes de preço',
         por_que='Os termos que mais separam o produto mal avaliado do bem avaliado são de qualidade física, não de preço.',
         indicador="o quanto 'noise' aparece mais em quem reclama, em pontos percentuais",
         valor_indicador='average +13,8 · bad +13,2 · noise +9,9 · battery +8,8 · bass +8,6',
         onde_ver='Customer insights'),
    dict(ordem=6,
         decisao='Tratar preço como alavanca de margem, não de satisfação',
         por_que='A nota ponderada é praticamente plana entre as faixas de preço. Quem compra barato não está mais insatisfeito.',
         indicador='amplitude da nota ponderada entre as seis faixas de preço',
         valor_indicador='amplitude de 0,10 ponto: de 4,113 a 4,216 nas seis faixas. OfficeProducts tem o menor desconto e a maior nota',
         onde_ver='Preços e descontos'),
])

dFonte = pd.DataFrame([
    dict(ordem=1, secao='Premissas',
         item='Taxa de câmbio', detalhe='A base só traz rupia indiana e não traz cotação. Toda conversão é assumida, com a data ao lado na tabela dMoeda.'),
    dict(ordem=2, secao='Premissas',
         item='Corte de percepção', detalhe='Percentis 25 e 75 do rating desta base: abaixo <= 3,90 e acima >= 4,30. Corte fixo em 3,0 marcaria 6 produtos de 1.351.'),
    dict(ordem=3, secao='Premissas',
         item='rating_count é proxy', detalhe='A base não traz volume vendido. rating_count mede engajamento, não venda. Conclusão sobre receita é inferência.'),
    dict(ordem=4, secao='Caminho',
         item='1. Origem', detalhe='Amazon Sales Dataset no Kaggle: 1.465 linhas, 16 colunas, preços em texto com o símbolo da rupia.'),
    dict(ordem=5, secao='Caminho',
         item='2. Limpeza', detalhe='Tipagem com cultura en-US, símbolo e separador de milhar removidos, percentual convertido em proporção.'),
    dict(ordem=6, secao='Caminho',
         item='3. Consolidação', detalhe='92 IDs repetidos resolvidos por desempate: menos nulos vence, empate decide pelo maior rating_count. Sobram 1.351 produtos.'),
    dict(ordem=7, secao='Caminho',
         item='4. Modelagem', detalhe='Star schema: fProdutos no grão de produto, dCategoria com a hierarquia em colunas, fAvaliacoes e as tabelas de termos.'),
    dict(ordem=8, secao='Limites',
         item='Amostra', detalhe='A base cobre um recorte do catálogo da Amazon na Índia, não o catálogo inteiro nem outro país.'),
    dict(ordem=9, secao='Limites',
         item='Concentração', detalhe='97,2% dos produtos estão em três categorias. Conclusão sobre categoria pequena não tem amostra que sustente.'),
    dict(ordem=10, secao='Limites',
         item='Nota comprimida', detalhe='O rating vai de 2,0 a 5,0 com mediana 4,10. Não existe produto mal avaliado em termos absolutos nesta base.'),
    dict(ordem=11, secao='Limites',
         item='Texto da avaliação', detalhe='review_title e review_content vêm colados por vírgula e o texto tem vírgula dentro. A contagem sai do review_id; o texto fica agregado por produto.'),
    dict(ordem=12, secao='Tratamentos',
         item='Categoria', detalhe='O campo é uma hierarquia separada por barra vertical, não uma lista. Virou coluna por nível, até 7. Explodir em linha inflaria toda soma em 4,2 vezes.'),
    dict(ordem=13, secao='Tratamentos',
         item='Chave da avaliação', detalhe='review_id não é chave: 759 ids aparecem em produtos diferentes. A tabela usa sk_avaliacao como chave substituta.'),
    dict(ordem=14, secao='Tratamentos',
         item='Termos', detalhe='Stopwords em inglês mais os genéricos do domínio. Termo presente em só uma faixa costuma ser nome de marca, então a comparação usa só os presentes nas duas.'),
    dict(ordem=15, secao='Tratamentos',
         item='Moeda', detalhe='dMoeda é tabela desconectada. Alimenta a segmentação e as medidas por SELECTEDVALUE, não filtra o fato.'),
])

print(f'dRecomendacao: {len(dRecomendacao)} decisões (texto autoral)')
print(f'dFonte: {len(dFonte)} itens em {dFonte.secao.nunique()} seções (texto autoral)')

## 8. Exportar para o Power BI

In [ ]:
saidas = {
    'fProdutos.csv':            fProdutos,
    'dCategoria.csv':           dCategoria,
    'fAvaliacoes.csv':          fAvaliacoes,
    'dTermosAvaliacao.csv':     dTermosAvaliacao,
    'dMoeda.csv':               dMoeda,
    'relatorio_duplicados.csv': relatorio_duplicados,
    'dTermoComparado.csv':      dTermoComparado,
    'dRecomendacao.csv':        dRecomendacao,
    'dFonte.csv':               dFonte,
}
for nome, tabela in saidas.items():
    tabela.to_csv(nome, index=False, encoding='utf-8-sig')
    print(f'{nome:<28} {len(tabela):>8,} linhas')

## 9. O modelo no Power BI

```
                  dMoeda (desconectada)
                        |
                   (só segmentação)

   dCategoria  1 ----- N  fProdutos  1 ----- N  fAvaliacoes
                              |
                              1
                              |
                              N
                      dTermosAvaliacao
```

Todos os relacionamentos com filtro cruzado **simples**, da dimensão para o fato:

| De | Para | Cardinalidade |
|---|---|---|
| `dCategoria[sk_categoria]` | `fProdutos[sk_categoria]` | 1 para N |
| `fProdutos[product_id]` | `fAvaliacoes[product_id]` | 1 para N |
| `fProdutos[product_id]` | `dTermosAvaliacao[product_id]` | 1 para N |
| `dMoeda` | nenhum | desconectada, de propósito |

A hierarquia de drill-down se monta arrastando `nivel_1` até `nivel_4` para dentro de uma
hierarquia em `dCategoria`. Do nível 5 em diante a amostra não sustenta (68, 19 e 7 caminhos).

### Medidas de moeda

```dax
Taxa selecionada =
SELECTEDVALUE( dMoeda[taxa_por_rupia], 1 )
```

```dax
Símbolo =
SELECTEDVALUE( dMoeda[simbolo], "₹" )
```

```dax
Preço com desconto =
SUM( fProdutos[discounted_price_rupia] ) * [Taxa selecionada]
```

```dax
Preço cheio =
SUM( fProdutos[actual_price_rupia] ) * [Taxa selecionada]
```

```dax
Desconto médio % =
AVERAGE( fProdutos[discount_percentage] )
```

> O desconto percentual **não** multiplica pela taxa. Percentual é adimensional, e
> multiplicar por câmbio é erro de unidade.

### Portfólio e engajamento

```dax
Produtos =
DISTINCTCOUNT( fProdutos[product_id] )
```

```dax
Avaliações =
SUM( fProdutos[rating_count] )
```

```dax
Avaliações com texto =
COUNTROWS( fAvaliacoes )
```

> `Avaliações` são 23,8 milhões, que é o `rating_count` declarado pela Amazon.
> `Avaliações com texto` são 10.598, que é o que a base realmente traz escrito.
> Os dois números são certos e medem coisas diferentes: dizer qual é qual evita a
> pergunta mais óbvia da banca.

```dax
Nota média ponderada =
DIVIDE(
    SUMX( fProdutos, fProdutos[rating] * fProdutos[rating_count] ),
    SUM( fProdutos[rating_count] ),
    BLANK()
)
```

> A média simples de `rating` trata um produto com 3 avaliações igual a um com 94 mil.
> A ponderada responde a pergunta de negócio. Vale mostrar as duas lado a lado numa tela:
> a diferença entre elas já é um insight.

### Perguntas 5 e 6 do enunciado

```dax
Alto engajamento, nota abaixo da base =
CALCULATE(
    [Produtos],
    FILTER(
        fProdutos,
        fProdutos[rating_count] >= 4740 && fProdutos[rating] <= 3.9
    )
)
```

```dax
Nota acima da base, pouca visibilidade =
CALCULATE(
    [Produtos],
    FILTER(
        fProdutos,
        fProdutos[rating] >= 4.3 && fProdutos[rating_count] < 1000
    )
)
```

> Os cortes 4740, 3,9, 4,3 e 1000 saíram da distribuição real: 4.740 é a mediana de
> `rating_count`, 3,9 é o p25 do `rating` e 4,3 é o p75. Não são números redondos
> escolhidos no olho, e isso é defensável no vídeo.

### Formatação do símbolo

Como a moeda muda com a segmentação, não dá para fixar `R$` no formato do visual:

1. **Formato dinâmico** no visual, em `Formato > Opções de formato dinâmico`, com `[Símbolo]`
2. Ou `[Símbolo]` no **título do cartão**, via título dinâmico com `fx`